In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("flax-community/papuGaPT2")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained("flax-community/papuGaPT2").to(device).eval()

/Users/bartek/Projects/courses-uwr/Modele Jezykowe/Pracownia2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1030 13:40:10.973000 91316 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 149/149 [00:00<00:00, 30353.65it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: flax-community/papuGaPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
@torch.no_grad()
def score(texts):
    """Log-prawdopodobieństwo całych tekstów (suma po tokenach)."""
    texts = [tokenizer.eos_token + t for t in texts]
    enc = tokenizer(texts, return_tensors="pt", padding=True).to(device)
    logits = model(**enc).logits[:, :-1]
    targets = enc.input_ids[:, 1:]
    logp = torch.log_softmax(logits, dim=-1).gather(2, targets.unsqueeze(2)).squeeze(2)
    return (logp * enc.attention_mask[:, 1:]).sum(1).tolist()

In [7]:
def disambiguate(text, beam_width=3):
    beams = [([], 0.0)]
    for variants in [w.split("|") for w in text.split()]:
        candidates = [words + [v] for words, _ in beams for v in variants]
        scores = score([" ".join(c) for c in candidates])
        beams = sorted(zip(candidates, scores), key=lambda x: -x[1])[:beam_width]
    return " ".join(beams[0][0])

In [8]:
text = (
    "wprost|wyprosty|wyprostu|wyprost "
    "uwielbiała|wielbił|wielbiła|uwielbił|wielbiło|uwielbiał|uwielbiało|uwielbiały "
    "słuchać|osłuchać|słychać|usłuchać "
    "o|i|e|a|ó|ę|y|ą|u "
    "wartościach "
    "własnych|owłosionych "
    "macierzy|mocarz|macierzą|macierze|mocarza|mocarze|mocarzy|macierz"
)

result = disambiguate(text)
print(result)

correct = [w.split("|")[0] for w in text.split()]
acc = sum(a == b for a, b in zip(result.split(), correct)) / len(correct)
print(f"Dokładność: {acc:.0%}")

wprost uwielbiał słuchać o wartościach własnych macierzy
Dokładność: 86%
